In [ ]:
import pandas as pd
import numpy as np
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, concordance_index_ipcw
import warnings
from tqdm import tqdm
import os
from sklearn.metrics import confusion_matrix

In [ ]:
version = 2

n = 100
# subset event level
# fp = f"./365day_future_prediction_outputs_50_subset_{n}_stage_filter_v{version}"
# full event level
fp = f"./365day_future_prediction_outputs_50_full_stage_filter_v{version}"

# subset patient level
# fp = "./365day_future_prediction_outputs_50_subset_1000_stage_filter_patient_level_v2"
# full patient level 
# fp = "./365day_future_prediction_outputs_50_full_stage_filter_patient_level_v2"

# baseline
fp = f"./365day_future_prediction_outputs_stage_filter_full_stage_filter_eskd_v{version}"
print(fp)

In [ ]:

os.listdir(fp)

In [ ]:
dirs = ['/XGBoost_365DayFuture_Classifier_detailed_outputs_classification.csv',
#  '/XGBoost_TTE_Survival_detailed_outputs_survival.csv',
#  'xgboost_only_365day_future_switch_analysis.csv'
 ]

# dirs = [
#     "/LSTM_365DayFutureTarget_detailed_outputs.csv",
#     "/MLP_365DayFutureTarget_detailed_outputs.csv",
#     "/RNN_365DayFutureTarget_detailed_outputs.csv",
#     "/TCN_365DayFutureTarget_detailed_outputs.csv",
#     "/Transformer_365DayFutureTarget_detailed_outputs.csv",
# ]

# dirs = [
#     "/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_RNN_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_TCN_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_Transformer_365DayFutureTarget_detailed_outputs.csv",
# ]

filepaths = [fp + i for i in dirs]
filepaths

In [ ]:
# df1 = pd.read_csv('./365day_future_prediction_outputs_50_subset_100_stage_filter_v1/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv').head()
# df1

In [ ]:
test_df = pd.read_csv(filepaths[0])

In [ ]:
test_df['cl_true_label'].unique(
)

In [ ]:
# # testing on df1
# df1['cl_true_label'].unique()

In [ ]:
# variable name, not metadata (results)
temp_results = filepaths[0]
test_meta = pd.read_csv(temp_results)

In [ ]:
test_meta.head()

In [ ]:
test_meta = test_meta.rename(columns={'EventDate': 'date',})

In [ ]:
test_meta.head()

In [ ]:
len(test_meta['PatientID'].unique())

In [ ]:

# %%
def select_encounters(df):
    """
        selects a single encounter row for each patient

        -For patients who progress in CKD stage (CKD_stage_numeric increases),
        selects the last encounter before the patient progresses in ckd stage
        -For patients whose CKD stage never changes, selects a random encounter.
        """
    df = df.copy()
    date_column = ""
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(by=['PatientID', 'date']).reset_index(drop=True)
    df['next_stage'] = df.groupby('PatientID')['CKD_stage_numeric'].shift(-1)
    # find changes in ckd stage
    df['last_encounter'] = df['next_stage'] > df['CKD_stage_numeric']

    selected_indices = []

    for patient, group in df.groupby('PatientID'):
        progressions = group[group['last_encounter']]

        if not progressions.empty:
            # patient progresses
            last_pre_progression_index = progressions.index.max()
            selected_indices.append(last_pre_progression_index)
        else:
            # patient does not progress
            random_index = np.random.choice(group.index)
            selected_indices.append(random_index)

    result_df = df.loc[selected_indices].drop(columns=['next_stage', 'last_encounter']).reset_index(drop=True)
    return result_df

test_meta = select_encounters(test_meta)

In [ ]:
test_meta

In [ ]:
len(test_meta['PatientID'].unique())

In [ ]:
output_dir = fp + "_patient_level"
print(output_dir)

In [ ]:
try:
    os.mkdir(output_dir)
except FileExistsError:
    pass

In [ ]:
test_file_name = dirs[0]
new_file_name = dirs[0].split(".")[0] + "_pt_lvl." + dirs[0].split(".")[1] 
print(output_dir)
print(new_file_name)
new_path = os.path.join(output_dir, new_file_name)
new_path = output_dir + new_file_name
print(new_path)


In [ ]:
test_meta.to_csv(new_path)